In [1]:
from google.colab import auth
auth.authenticate_user()

In [8]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
from google.colab import drive
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
DATASET_ID = "produccion"
TABLE_ID= "TABLA_Anulaciones"

ARCHIVO_EXCEL= "/content/drive/MyDrive/BRIMAC_FCR1_CONTROL_ENERO_2024-2025.xlsx"

In [21]:
bigquery_client = bigquery.Client(project=PROJECT_ID)

schema_anulaciones = [
    bigquery.SchemaField("SITE", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("FECHA_INGRESO", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("MES_DECLARADO", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("ANIO_DECLARADO", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("FECHA_CONCATENADA_DECLARADO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("HORA_INGRESO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("APELLIDO_PATERNO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("APELLIDO_MATERNO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("NOMBRES", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("TIPO_DOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("NRO_DOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("TELEFONO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("CERT_BANCO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("NRO_CASO", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("CERT_BANCO2", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("NRO_CUENTA_DESTINO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.FLOAT),
    bigquery.SchemaField("TIPO_SEGURO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("TIPOLOGIA", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("NUMERO_PRIMAS_SOLICITADAS", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("DIVISA", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("IMPORTE_OPERACION", bigquery.enums.SqlTypeNames.FLOAT),
    bigquery.SchemaField("PEDIDO_DETALLE_ERROR", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("REGISTRO_ASESOR", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("PROCEDE", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("IMPORTE_ABONADO_AL_CLIENTE", bigquery.enums.SqlTypeNames.FLOAT),
    bigquery.SchemaField("DIVISA_DE_ABONO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("NRO_DE_PRIMAS_DEVUELTAS", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("DIVISA_ORIGINAL_PRIMA", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("IMPORTE_DIVISA_ORIG_PRIMAS_DEVUELTAS", bigquery.enums.SqlTypeNames.FLOAT),
    bigquery.SchemaField("CIA", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("FECHA_ALTA", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("CUMPLE_CRITERIOS", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("REGISTRO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("GLOSA", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("CONTRATO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("OBS_RIMAC", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("RSPTA_EJECUTIVO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("STATUS", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("ESTADO_CARINA_SALDANA_OMX", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("OBSERV_LEVANTADAS_BANCO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("ESTADO_GENERAL", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("OBSERVACIONES", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("COMENTARIO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("ESTADO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("LA", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("IMPORTE_LA_GENERADA", bigquery.enums.SqlTypeNames.FLOAT),
    bigquery.SchemaField("ESTADO_DE_PAGO", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("FECHA_DE_EMISION_LA", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("ANIO_ANU", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("MES_ANU", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("FECHA_CONCATENADA_EMI_LA", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("FECHA_DE_ABONO", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("ESTADO_FINAL", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("ANIO_ANULACION", bigquery.enums.SqlTypeNames.FLOAT),
    bigquery.SchemaField("ESTADO_FINAL_REPORTE_K", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("RESPONSABLE", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("FECHA_ENVIO_EQUIPO", bigquery.enums.SqlTypeNames.DATETIME),
]

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    try:
      tabla = bigquery_client.get_table(table_ref)
    except:
        tabla_tramas = bigquery.Table(table_ref, schema=schema)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        print(f'ℹ️ ----- Se ha creado la tabla: {table_id} en el dataset: {dataset_id} -----')
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    return

df_anulaciones= pd.read_excel(ARCHIVO_EXCEL,
                             sheet_name='BASE_2024-2025', dtype={'N° Documento':str, 'Cert banco':str,
                                                                 'Cert banco2':str,'N° Cuenta Destino': str,
                                                                 'Teléfono':str, '¿CUMPLE CRITERIOS?': str,
                                                                 'Registro':str, 'contrato':str,
                                                                 'Hora Ingreso':str})
df_anulaciones = df_anulaciones[df_anulaciones.columns[:58]]
df_anulaciones.columns = (df_anulaciones.columns.str.strip()
                          .str.upper()  # opcional: todo en mayúsculas
                          .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                        )

df_anulaciones= df_anulaciones.rename(columns={'A_O_DECLARADO': 'ANIO_DECLARADO',
                                               'FECHA_CONCATENADA_DECLARACI_N': 'FECHA_CONCATENADA_DECLARADO',
                                               'N__DOCUMENTO': 'NRO_DOCUMENTO', 'TEL_FONO':'TELEFONO',
                                               'N__CASO':'NRO_CASO', 'N__CUENTA_DESTINO':'NRO_CUENTA_DESTINO',
                                               'N_MERO_PRIMAS_SOLICITADAS':'NUMERO_PRIMAS_SOLICITADAS',
                                               'IMPORTE_OPERACI_N':'IMPORTE_OPERACION',
                                               'N__DE_PRIMAS_DEVUELTAS':'NRO_DE_PRIMAS_DEVUELTAS',
                                               'IMPORTE_DIVISA_ORIGINAL_DE_PRIMAS_DEVUELTAS':'IMPORTE_DIVISA_ORIG_PRIMAS_DEVUELTAS',
                                               '_CUMPLE_CRITERIOS_':'CUMPLE_CRITERIOS',
                                               'OBS RIMAC':'OBSERV_RIMAC',
                                               'ESTADO_CARINA_SALDA_A_OMX':'ESTADO_CARINA_SALDANA_OMX',
                                               'OBSERVACIONES_LEVANTADAS_DEL_BANCO':'OBSERV_LEVANTADAS_BANCO',
                                               'COMENTARIO_R':'COMENTARIO', 'A_O_ANU':'ANIO_ANU',
                                               'IMPORTE_DE_LA_GENERADA':'IMPORTE_LA_GENERADA',
                                               'CONCATENADO_EMI_LA_S':'FECHA_CONCATENADA_EMI_LA',
                                               'A_O_ANULACION':'ANIO_ANULACION','FECHA_DE_ENVIO_EQUIPO':'FECHA_ENVIO_EQUIPO',
                                               })

df_anulaciones.loc[df_anulaciones['TIPO_DOCUMENTO'] == 'CARNET DE EXTRANJERÍA', 'TIPO_DOCUMENTO'] = 'CE'

df_anulaciones['TIPO_SEGURO']= df_anulaciones['TIPO_SEGURO'].str.upper()
df_anulaciones['TIPO_DOCUMENTO']= df_anulaciones['TIPO_DOCUMENTO'].str.upper()
df_anulaciones['REGISTRO_ASESOR']= df_anulaciones['REGISTRO_ASESOR'].str.upper()
df_anulaciones['PROCEDE']= df_anulaciones['PROCEDE'].str.upper()
df_anulaciones['DIVISA_DE_ABONO']= df_anulaciones['DIVISA_DE_ABONO'].str.upper()
df_anulaciones['ESTADO_FINAL_REPORTE_K']= df_anulaciones['ESTADO_FINAL_REPORTE_K'].str.upper()
df_anulaciones['DIVISA_ORIGINAL_PRIMA']= df_anulaciones['DIVISA_ORIGINAL_PRIMA'].str.upper()
df_anulaciones['CIA']= df_anulaciones['CIA'].str.upper()
df_anulaciones['SITE'] = df_anulaciones['SITE'].astype(str)
df_anulaciones['APELLIDO_PATERNO'] = df_anulaciones['APELLIDO_PATERNO'].astype(str)
df_anulaciones['APELLIDO_MATERNO'] = df_anulaciones['APELLIDO_MATERNO'].astype(str)
df_anulaciones['NOMBRES'] = df_anulaciones['NOMBRES'].astype(str)
df_anulaciones['TIPO_DOCUMENTO'] = df_anulaciones['TIPO_DOCUMENTO'].astype(str)
df_anulaciones['NRO_DOCUMENTO'] = df_anulaciones['NRO_DOCUMENTO'].astype(str)
df_anulaciones['PEDIDO_DETALLE_ERROR'] = df_anulaciones['PEDIDO_DETALLE_ERROR'].astype(str)
df_anulaciones['OBSERV_LEVANTADAS_BANCO'] = df_anulaciones['OBSERV_LEVANTADAS_BANCO'].astype(str)
df_anulaciones['LA'] = df_anulaciones['LA'].astype(str)
df_anulaciones['RESPONSABLE'] = df_anulaciones['RESPONSABLE'].astype(str)
df_anulaciones['NUMERO_PRIMAS_SOLICITADAS'] = df_anulaciones['NUMERO_PRIMAS_SOLICITADAS'].fillna(0).astype('int')
df_anulaciones['IMPORTE_OPERACION'] = pd.to_numeric(df_anulaciones['IMPORTE_OPERACION'], errors="coerce").astype('float64')
df_anulaciones['FECHA_ALTA'] = pd.to_datetime(df_anulaciones['FECHA_ALTA'], format='%d/%m/%Y', errors='coerce')
df_anulaciones['IMPORTE_LA_GENERADA'] = pd.to_numeric(df_anulaciones['IMPORTE_LA_GENERADA'], errors="coerce").astype('float64')
df_anulaciones['FECHA_ENVIO_EQUIPO'] = pd.to_datetime(df_anulaciones['FECHA_ENVIO_EQUIPO'], format='%d/%m/%Y', errors='coerce')

Guardar_en_BigQuery(df_anulaciones, DATASET_ID, TABLE_ID, schema_anulaciones)
print(f"✅ ARCHIVO CARGADO ...")

✅ ARCHIVO CARGADO ...


In [6]:
df_anulaciones= pd.read_excel(ARCHIVO_EXCEL,
                             sheet_name='BASE_2024-2025', dtype={'N° Documento':str, 'Cert banco':str,
                                                                 'Cert banco2':str,'N° Cuenta Destino': str,
                                                                 'Teléfono':str, '¿CUMPLE CRITERIOS?': str,
                                                                 'Registro':str, 'contrato':str,
                                                                 'Hora Ingreso':str})

In [16]:
df_anulaciones = df_anulaciones[df_anulaciones.columns[:58]]
df_anulaciones.columns = (df_anulaciones.columns.str.strip()
                          .str.upper()  # opcional: todo en mayúsculas
                          .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                        )

In [17]:
df_anulaciones= df_anulaciones.rename(columns={'A_O_DECLARADO': 'ANIO_DECLARADO',
                                               'FECHA_CONCATENADA_DECLARACI_N': 'FECHA_CONCATENADA_DECLARADO',
                                               'N__DOCUMENTO': 'NRO_DOCUMENTO', 'TEL_FONO':'TELEFONO',
                                               'N__CASO':'NRO_CASO', 'N__CUENTA_DESTINO':'NRO_CUENTA_DESTINO',
                                               'N_MERO_PRIMAS_SOLICITADAS':'NUMERO_PRIMAS_SOLICITADAS',
                                               'IMPORTE_OPERACI_N':'IMPORTE_OPERACION',
                                               'N__DE_PRIMAS_DEVUELTAS':'NRO_DE_PRIMAS_DEVUELTAS',
                                               'IMPORTE_DIVISA_ORIGINAL_DE_PRIMAS_DEVUELTAS':'IMPORTE_DIVISA_ORIG_PRIMAS_DEVUELTAS',
                                               '_CUMPLE_CRITERIOS_':'CUMPLE_CRITERIOS',
                                               'OBS RIMAC':'OBSERV_RIMAC',
                                               'ESTADO_CARINA_SALDA_A_OMX':'ESTADO_CARINA_SALDANA_OMX',
                                               'OBSERVACIONES_LEVANTADAS_DEL_BANCO':'OBSERV_LEVANTADAS_BANCO',
                                               'COMENTARIO_R':'COMENTARIO', 'A_O_ANU':'ANIO_ANU',
                                               'IMPORTE_DE_LA_GENERADA':'IMPORTE_LA_GENERADA',
                                               'CONCATENADO_EMI_LA_S':'FECHA_CONCATENADA_EMI_LA',
                                               'A_O_ANULACION':'ANIO_ANULACION','FECHA_DE_ENVIO_EQUIPO':'FECHA_ENVIO_EQUIPO',
                                               })

In [18]:
df_anulaciones.head(3)

,SITE,FECHA_INGRESO,MES_DECLARADO,ANIO_DECLARADO,FECHA_CONCATENADA_DECLARADO,HORA_INGRESO,APELLIDO_PATERNO,APELLIDO_MATERNO,NOMBRES,TIPO_DOCUMENTO,NRO_DOCUMENTO,TELEFONO,CERT_BANCO,NRO_CASO,CERT_BANCO2,NRO_CUENTA_DESTINO,PRODUCTO,TIPO_SEGURO,TIPOLOGIA,NUMERO_PRIMAS_SOLICITADAS,DIVISA,IMPORTE_OPERACION,PEDIDO_DETALLE_ERROR,REGISTRO_ASESOR,PROCEDE,IMPORTE_ABONADO_AL_CLIENTE,DIVISA_DE_ABONO,NRO_DE_PRIMAS_DEVUELTAS,DIVISA_ORIGINAL_PRIMA,IMPORTE_DIVISA_ORIG_PRIMAS_DEVUELTAS,CIA,FECHA_ALTA,CUMPLE_CRITERIOS,REGISTRO,GLOSA,CONTRATO,OBSERV_RIMAC,RSPTA_EJECUTIVO,STATUS,ESTADO_CARINA_SALDANA_OMX,OBSERV_LEVANTADAS_BANCO,ESTADO_GENERAL,OBSERVACIONES,COMENTARIO,ESTADO,LA,IMPORTE_LA_GENERADA,ESTADO_DE_PAGO,FECHA_DE_EMISION_LA,ANIO_ANU,MES_ANU,FECHA_CONCATENADA_EMI_LA,FECHA_DE_ABONO,ESTADO_FINAL,ANIO_ANULACION,ESTADO_FINAL_REPORTE_K,RESPONSABLE,FECHA_ENVIO_EQUIPO
0,21665,2024-01-01,1,2024,2024-1,13:05:39,ARTEAGA,DIAZ X,RUBEN DARIO,Carnet de extranjería,005720985,982285433,00110178184000382668,20,00110178184000382668,4147918367114211,NaN,SALUD A TU ALCANCE,a una compañera,NaN,Sol,1.138,CTA: 0011-0178-18-4000382668 DIV: SOLES ...,xp55750,Procede,1138.00,Soles,1,Soles,1138.0,Rimac,2023-12-29 00:00:00,NaN,NaN,ST21665SATA01784000382668,NaN,NaN,NaN,NaN,NaN,NaN,CONCLUIDO,ABONADO,CONCILIADO,CASOS CERRADO BANCO,166747568,1137.99,ABONADO,2024-01-12,2024,1,2024-1,2024-01-22,CONCLUIDO,2024.0,ANULADO,NaN,NaN
1,21668,2024-01-02,1,2024,2024-1,13:05:39,MUSTTO,GARCIA,EGLON DANIEL,DNI,75979838,982822043,00110183164000988164,20,00110183164000988164,0011-0579-00-0204366216,NaN,PROTECCIÓN DE TARJETA,Devolución de prima por renovacion Anual,NaN,Dolar,41,P.P. - - - N.MOV: 9400300...,xp69068,Procede,176.53,Soles,1,Dólares,41.0,Rimac,2022-12-19 00:00:00,NaN,NaN,ST21668PT01834000988164,NaN,NaN,NaN,NaN,NaN,NaN,CONCLUIDO,ABONADO,CONCILIADO,CASOS CERRADO BANCO,166852029,41,ABONADO,2024-01-17,2024,1,2024-1,2024-01-22,CONCLUIDO,2024.0,ANULADO,NaN,NaN
2,21669,2024-01-02,1,2024,2024-1,13:05:39,SHIMABUKURO,MOROMISATO,JIMMY MARTIN,Ruc,2042707520,981900203,00110194874000475663,20,00110194874000475663,0011-0148-00-0100050698,NaN,MULTIRIESGO NEGOCIO,Devolución de prima por renovacion Anual,NaN,Sol,2294,CONSULTA MOVIMIENTOS - SEGUROS ...,XP67320,Procede,2294.00,Soles,1,Soles,2294.0,Rimac,2022-12-29 00:00:00,NaN,NaN,ST21669MN01944000475663,NaN,NaN,NaN,NaN,NaN,NaN,CONCLUIDO,ABONADO,CONCILIADO,CASOS CERRADO BANCO,166852554,2293.99,ABONADO,2024-01-17,2024,1,2024-1,2024-01-22,CONCLUIDO,2024.0,ANULADO,NaN,NaN


In [19]:
df_anulaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35435 entries, 0 to 35434
Data columns (total 58 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   SITE                                  35435 non-null  object        
 1   FECHA_INGRESO                         35435 non-null  datetime64[ns]
 2   MES_DECLARADO                         35435 non-null  int64         
 3   ANIO_DECLARADO                        35435 non-null  int64         
 4   FECHA_CONCATENADA_DECLARADO           35435 non-null  object        
 5   HORA_INGRESO                          35435 non-null  object        
 6   APELLIDO_PATERNO                      35435 non-null  object        
 7   APELLIDO_MATERNO                      35435 non-null  object        
 8   NOMBRES                               35435 non-null  object        
 9   TIPO_DOCUMENTO                        35435 non-null  object        
 10